# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [76]:
ПРІЗВИЩЕ_ІМЯ = 'Колбасюк Данило'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 5       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [77]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json','r',encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 5   ·   Колбасюк Данило   ·   ШІ-32

Ваші вісім стовпців:
   1. Телефон
   2. помер/жив
   3. ЛН
   4. Виписка
   5. spO2 без дот
   6. Прокальцитонін
   7. БРнепр1
   8. Гематокрит1

Ваша пара для аналізу зв’язку: Стать та ступінь КТ

Ваше додаткове питання:
   Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?


In [78]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,помер/жив,ЛН,Виписка,spO2 без дот,Прокальцитонін,БРнепр1,Гематокрит1
0,0958469788 дружина Ганна,1,ІІІ-IV,2022-03-10 00:00:00,90.0,0.126,10.2,48.0
1,"0671778486, 0994887663 дружина Оксана",2,ІІ-ІІІ,2022-03-16 00:00:00,83.0,NaN,1.8,42.2
2,"0958768045, чоловік Степан 0951784726",2,ІІ,2022-01-18 00:00:00,78.0,NaN,NaN,40.0
3,0959819331. дочка Валентина 0960845205,2,ІІІ-IV,2022-01-13 00:00:00,82.0,NaN,8.5,22.0
4,"0952294866, дочка Дарина 0678360161",2,ІІ-ІІІ,2022-03-11 00:00:00,80.0,NaN,10.2,40.0
5,0959922983 сестра Софія,1,ІІ-ІІІ,2021-12-26 00:00:00,72.0,NaN,8.5,39.0
6,"0998073702, дружина Наталія 0998382226",2,ІІ-ІІІ,2022-05-11 00:00:00,89.0,NaN,10.2,44.9
7,"0637258771, син Сергій 0968413301",2,І,2022-05-28 00:00:00,90.0,NaN,8.5,38.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [79]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Телефон         120 non-null    object 
 1   помер/жив       121 non-null    int64  
 2   ЛН              121 non-null    str    
 3   Виписка         120 non-null    object 
 4   spO2 без дот    120 non-null    float64
 5   Прокальцитонін  46 non-null     float64
 6   БРнепр1         108 non-null    float64
 7   Гематокрит1     115 non-null    float64
dtypes: float64(4), int64(1), object(2), str(1)
memory usage: 8.6+ KB


In [80]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Прокальцитонін
    непорожніх 46 з 121 | унікальних 17 | типи: float×46
    прик

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [81]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'Телефон': ('ідентифікатор', 'всі записи унікальні, тому їх можна використовувати як ключ'),
    'помер/жив': ('номінальна', 'числа не є кількістю, а лише стани'),
    'ЛН': ('порядкова', 'числа є порядком, але не знаємо різниці між ними'),
    'Виписка': ('інтервальна', 'записи - дати, для яких можна обчислювати різницю, але не відомо, чи є нульова точка'),
    'spO2 без дот': ('відношень', 'вимірювані значення, які можемо порівнювати'),
    'Прокальцитонін': ('відношень', 'вимірювані значення, які можемо порівнювати'),
    'БРнепр1': ('відношень', 'вимірювані значення, які можемо порівнювати'),
    'Гематокрит1': ('відношень', 'вимірювані значення, які можемо порівнювати')
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"всі записи унікальні, тому їх можна використовувати як ключ"
1,помер/жив,номінальна,"числа не є кількістю, а лише стани"
2,ЛН,порядкова,"числа є порядком, але не знаємо різниці між ними"
3,Виписка,інтервальна,"записи - дати, для яких можна обчислювати різницю, але н..."
4,spO2 без дот,відношень,"вимірювані значення, які можемо порівнювати"
5,Прокальцитонін,відношень,"вимірювані значення, які можемо порівнювати"
6,БРнепр1,відношень,"вимірювані значення, які можемо порівнювати"
7,Гематокрит1,відношень,"вимірювані значення, які можемо порівнювати"


In [82]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [83]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()
    if ш in ('ідентифікатор'):
        return р
    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації
    р['мода'] = з.mode().iloc[0] if not з.mode().empty else None
    р['частоти'] = з.value_counts().to_dict()
    if ш == 'порядкова':
        if стовпець == 'ЛН':
            return р
        elif pd.api.types.is_numeric_dtype(з):
            р['медіана'] = з.median()
            р['квантилі'] = з.quantile(
                [0.25, 0.5, 0.75]
            ).to_dict()
    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if pd.api.types.is_numeric_dtype(з):
            р['середнє'] = з.mean()
            р['стандартне відхилення'] = з.std()
    if ш in ('відношень', 'абсолютна'):
        if pd.api.types.is_numeric_dtype(з):
            середнє = з.mean()
            if середнє != 0:
                р['коефіцієнт варіації'] = з.std() / середнє
            else:
                р['коефіцієнт варіації'] = None

    return р

print ('Для стовпця ЛН рахується тільки частота, бо немає числового представлення')
for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

Для стовпця ЛН рахується тільки частота, бо немає числового представлення
--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- помер/жив  (номінальна)
    {'непорожніх': 121, 'унікальних': 2, 'мода': np.int64(1), 'частоти': {1: 61, 2: 60}}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'частоти': {'ІІ-ІІІ': 59, 'ІІ': 21, 'ІІІ': 17, 'І': 12, 'І-ІІ': 9, 'ІІІ-IV': 3}}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': datetime.datetime(2021, 9, 7, 0, 0), 'частоти': {datetime.datetime(2021, 11, 19, 0, 0): 3, datetime.datetime(2021, 9, 7, 0, 0): 3, datetime.datetime(2021, 9, 15, 0, 0): 3, datetime.datetime(2021, 9, 11, 0, 0): 3, datetime.datetime(2022, 4, 6, 0, 0): 3, datetime.datetime(2022, 6, 6, 0, 0): 3, datetime.datetime(2022, 1, 18, 0, 0): 2, datetime.datetime(2022, 5, 11, 0, 0): 2, datetime.datetime(2022, 5, 13, 0, 0): 2, datetime.datetime(2021, 11, 16, 0, 0): 2, datetime.datetime(2022, 5, 12, 0, 0): 2

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [84]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,0.502062,1.0,1.0,1.0,2.0,2.0
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,12.208249,0.9,78.75,84.0,89.0,98.0
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
БРнепр1,108.0,NaN,NaN,NaN,10.24537,2.398924,1.8,8.5,10.2,11.35,22.1
Гематокрит1,115.0,NaN,NaN,NaN,39.092174,7.324773,10.0,36.0,40.0,43.5,56.0


In [85]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Телефон', 'unique', 120, 'ідентифікатор, всі записи унікальні'),
    ('помер/жив', 'mean', 1.495868, 'номінальна шкала, а самі значення - коди'),
    ('помер/жив', 'std', 0.502062, 'номінальна шкала, а самі значення - коди')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,unique,120.000000,"ідентифікатор, всі записи унікальні"
1,помер/жив,mean,1.495868,"номінальна шкала, а самі значення - коди"
2,помер/жив,std,0.502062,"номінальна шкала, а самі значення - коди"


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [86]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × ступінь КТ

Стать: шкала — не у вашому списку
ступінь КТ: шкала — не у вашому списку

  Стать ступінь КТ
0     ч    сердній
1     ч   середній
2     ж  критичний
3     ж  критичний
4     ж        NaN
5     ж   середній


In [87]:
from scipy import stats
ОБРАНА_МІРА = 'коефіцієнт Крамера (V)'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'стать є номінальною шкалою, з 2 унікальними значеннями, які є кодами, а ступінь КТ - порядкова шкала, бо є порядок в заданих значеннях, але ми різниці між ними не маємо, саме тому підходить тільки коефіцієнт Крамера (V)'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
дані=df[[A,B]].dropna()
таблиця = pd.crosstab(дані[A], дані[B])
chi2, p, dof, expected = stats.chi2_contingency(таблиця)
n=таблиця.to_numpy().sum()
r,c=таблиця.shape
V=np.sqrt(chi2/(n*(min(r-1,c-1))))
print(f"V = {V:.4f}")

V = 0.6042


In [88]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
стать_число = df['Стать'].map({
    'ч': 0,
    'ж': 1
    })

кт_число = df['ступінь КТ'].map({
    'середній': 1,
    'критичний': 2
})
кореляція = стать_число.corr(кт_число)
print('Кореляція Пірсона:', кореляція)
# Стать є номінальною шкалою, які ми можемо кодувати як хочемо, і через це постійно змінюватиметься знак кореляції, тому нам Пірсона не підходить.

Кореляція Пірсона: 0.7071067811865474


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [89]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'Телефон': 'не берем у модель',
    'помер/жив': 'one-hot кодування, 0 - помер, 1 - жив',
    'ЛН': 'задаємо цілі коди від 1 до 6 для 6 присутніх категорій',
    'Виписка': 'дати, переходимо до тривалостей',
    'spO2 без дот': 'нормалізація Min-Max до діапазону [0, 1]',
    'Прокальцитонін': 'нормалізація Min-Max до діапазону [0, 1]',
    'БРнепр1': 'нормалізація Min-Max до діапазону [0, 1]',
    'Гематокрит1': 'нормалізація Min-Max до діапазону [0, 1]'
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,не берем у модель
1,помер/жив,номінальна,"one-hot кодування, 0 - помер, 1 - жив"
2,ЛН,порядкова,задаємо цілі коди від 1 до 6 для 6 присутніх категорій
3,Виписка,інтервальна,"дати, переходимо до тривалостей"
4,spO2 без дот,відношень,"нормалізація Min-Max до діапазону [0, 1]"
5,Прокальцитонін,відношень,"нормалізація Min-Max до діапазону [0, 1]"
6,БРнепр1,відношень,"нормалізація Min-Max до діапазону [0, 1]"
7,Гематокрит1,відношень,"нормалізація Min-Max до діапазону [0, 1]"


In [90]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
дані = df[['помер/жив', 'ЛН', 'spO2 без дот']].copy()
дані['помер/жив'] = дані['помер/жив'].map({
    1: 0,
    2: 1
})
порядок_ЛН = {
    'І': 1,
    'І-ІІ': 2,
    'ІІ': 3,
    'ІІ-ІІІ': 4,
    'ІІІ': 5,
    'ІІІ-IV': 6
}
дані['ЛН_код'] = дані['ЛН'].map(порядок_ЛН)

мін=дані['spO2 без дот'].min()
макс=дані['spO2 без дот'].max()
дані['spO2_норм'] = (
    (дані['spO2 без дот'] - мін)/(макс-мін)
)
підготовлені_дані = дані[
    ['помер/жив', 'ЛН_код', 'spO2_норм']
]

print(підготовлені_дані.head(10))

   помер/жив  ЛН_код  spO2_норм
0          0       6   0.917611
1          1       4   0.845520
2          1       3   0.794027
3          1       6   0.835221
4          1       4   0.814624
5          0       4   0.732235
6          1       4   0.907312
7          1       1   0.917611
8          1       3   0.794027
9          0       5   0.814624


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [91]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [92]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [93]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (1,'ми не можемо порахувати середній результат лікування','шкала для цього стовпця - номінальна, тому для неї ми середнє не можемо обчислити','замість цього можемо показати, який відсоток пацієнтів помер, а який вижив'),
    (2,'в нас є різні представлення одної статі','перед тим як вказати найбільшу групу, не було переведено дані до одного формату, тому деякі групи виглядають помітно менші, але насправді є частинами більших груп, просто з іншою назвою','перевести представлення до одного формату, наприклад жін до ж та чол до ч'),
    (3,'порушено порядкова шкала тяжкості ураження легень, а також некоректне представлення середнього ступеня ураження','тут розподіл показує частоти цих ступенів, а не тяжкість, плюс є декілька різних представлень одного ступеня, також тут навіть не за зростанням показано розподіл, а за алфавітним порядком назви стовпців',"об'єднати різні представлення одного ступеня в один, сортувати саме за частотою"),
    (4,'в стовпці КТ не всі значення представлені одинаково','всі дані потипу 20-30% перетворюються на NaN після pd.to_numeric, що зменшує вибірку','можемо для таких діапазонів просто взяти середину'),
    (5,'різниця між мінімумом та максимумом дуже велика','всі записи представлені у відсотках, але у всіх записах крім одного немає символу %, тому вони лишаються як є, а запис 90% став 0.9','привести всі записи до одного формату'),
    (6,'застосовано коефіцієнт варіації до дат, які були представлені як кількості днів від 1970 року','це число залежить від обраної нульвої точки','розглянути діапазон дат замість самих дат'),
    (7,'застосовано коефіцієнт Пірсона','стать та летальність мають номінальну шкалу, для них не підходить коефіцієнт Пірсона','взяти χ² та коефіцієнт Крамера'),
    (8,'не всі стовпці потрібно нормалізовувати','вік не потрібно нормалізовувати, бо ми не знаємо який мінімум та максимум у віці після нормалізації, для ФЕРИТ2 значення тільки одне, тому нормалізовувати його немає сенсу','вік та ФЕРИТ2 не нормалізувати'),
    (9,'всі рядки зникли','всі рядки в наборі мають хоча б один пустий стовпець, тому всі просто видаляються','перевіряти пропуски тільки для деяких ознак'),
    (10,'не можна просто так вказати дату розтину як найважливішу ознаку','дата розтину фактично тільки присутня для летальних випадків, тому показник завжди буде 1','не брати дату розтину до моделі')
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,ми не можемо порахувати середній результат лікування,"шкала для цього стовпця - номінальна, тому для неї ми се...","замість цього можемо показати, який відсоток пацієнтів п..."
1,2,в нас є різні представлення одної статі,"перед тим як вказати найбільшу групу, не було переведено...","перевести представлення до одного формату, наприклад жін..."
2,3,"порушено порядкова шкала тяжкості ураження легень, а так...","тут розподіл показує частоти цих ступенів, а не тяжкість...","об'єднати різні представлення одного ступеня в один, сор..."
3,4,в стовпці КТ не всі значення представлені одинаково,всі дані потипу 20-30% перетворюються на NaN після pd.to...,можемо для таких діапазонів просто взяти середину
4,5,різниця між мінімумом та максимумом дуже велика,"всі записи представлені у відсотках, але у всіх записах ...",привести всі записи до одного формату
5,6,"застосовано коефіцієнт варіації до дат, які були предста...",це число залежить від обраної нульвої точки,розглянути діапазон дат замість самих дат
6,7,застосовано коефіцієнт Пірсона,"стать та летальність мають номінальну шкалу, для них не ...",взяти χ² та коефіцієнт Крамера
7,8,не всі стовпці потрібно нормалізовувати,"вік не потрібно нормалізовувати, бо ми не знаємо який мі...",вік та ФЕРИТ2 не нормалізувати
8,9,всі рядки зникли,"всі рядки в наборі мають хоча б один пустий стовпець, то...",перевіряти пропуски тільки для деяких ознак
9,10,не можна просто так вказати дату розтину як найважливішу...,дата розтину фактично тільки присутня для летальних випа...,не брати дату розтину до моделі


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [94]:
# ваш код: три правильні обчислення замість хибних
print("Перший розділ")
середній_результат = df['помер/жив'].mean()
print(f'Середній результат лікування: {середній_результат:.4f}')
помер=df['помер/жив'].value_counts(normalize=True).get(1, 0)
print(f'Відсоток померлих: {помер:.4}')
жив=df['помер/жив'].value_counts(normalize=True).get(2, 0)
print(f'Відсоток живих: {жив:.4}')
print("Другий розділ")
стать = df['Стать'].value_counts()
print('У вибірці представлено', df['Стать'].nunique(), 'статі:')
print(стать.to_string())
df['Стать'] = df['Стать'].replace({
    'чол': 'ч',
    'жін': 'ж'
})
стать = df['Стать'].value_counts()
print('У вибірці представлено', df['Стать'].nunique(), 'статі:')
print(стать.to_string())
s = df['spO2 без дот']
print('Сатурація при поступленні без дотації кисню')
print('  середнє      :', round(s.mean(), 2))
print('  медіана      :', round(s.median(), 2))
print('  мінімум      :', round(s.min(), 2))
print('  максимум     :', round(s.max(), 2))
print()
df['spO2 без дот'] = df['spO2 без дот'].replace({
    0.9: 90
})
print('Сатурація при поступленні без дотації кисню')
print('  середнє      :', round(s.mean(), 2))
print('  медіана      :', round(s.median(), 2))
print('  мінімум      :', round(s.min(), 2))
print('  максимум     :', round(s.max(), 2))
print()

Перший розділ
Середній результат лікування: 1.4959
Відсоток померлих: 0.5041
Відсоток живих: 0.4959
Другий розділ
У вибірці представлено 4 статі:
Стать
ж      56
ч      34
жін    19
чол    12
У вибірці представлено 2 статі:
Стать
ж    75
ч    46
Сатурація при поступленні без дотації кисню
  середнє      : 82.05
  медіана      : 84.0
  мінімум      : 0.9
  максимум     : 98.0

Сатурація при поступленні без дотації кисню
  середнє      : 82.05
  медіана      : 84.0
  мінімум      : 0.9
  максимум     : 98.0



---
# Етап 7. Висновки та індивідуальне питання

In [95]:
ВИСНОВКИ = [
    'немає рядка, в якому всі стовпці заповнені, просто dropna() використати не можна',
    'кількість померлих та живих пацієнтів майже однакова, з різницею тільки у піввідсотка',
    'для ідентифікаторів можна використати телефон, а також номер і/x'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. немає рядка, в якому всі стовпці заповнені, просто dropna() використати не можна
2. кількість померлих та живих пацієнтів майже однакова, з різницею тільки у піввідсотка
3. для ідентифікаторів можна використати телефон, а також номер і/x


In [96]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = 'Стать та результат лікування мають номінальну шкалу (тільки два унікальних значення). Коефіцієнт кореляції Пірсона підходить тільки для ознак з відносною шкалою. Для цієї пари можна взяти χ² та коефіцієнт Крамера'
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?

Стать та результат лікування мають номінальну шкалу (тільки два унікальних значення). Коефіцієнт кореляції Пірсона підходить тільки для ознак з відносною шкалою. Для цієї пари можна взяти χ² та коефіцієнт Крамера


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [97]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = "розподіл ознак на різні шкали, деякі пояснення розділів для виправлення аналізу, правильне написання мір зв'зків, виправлення коду"
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [98]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 5 готова до захисту.
Колбасюк Данило, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.